# Scraper AhorraMás

Este notebook sigue la misma lógica que los scrapers de **Mercadona**, **DIA** y **Alcampo**:

1. Parte siempre de la página principal de AhorraMás.
2. Localiza y acepta las cookies de forma automática.
3. Navega directamente a la URL de búsqueda de cada producto.
4. Hace scroll dinámico para cargar todos los productos disponibles en la página de resultados.
5. Extrae el nombre, precio, URL e identificador de cada producto usando BeautifulSoup.
6. Une todas las búsquedas en un único dataset.
7. Elimina duplicados por `product_id` y nombre.
8. Guarda:
   - un único CSV `raw` en `data/raw`
   - un único CSV `processed` en `data/processed`

**Primero se ejecuta en modo prueba con `leche`.**  
Cuando funcione correctamente, cambia `TEST_MODE = False` para lanzar todas las búsquedas de la lista.

---

### Qué comprobar en la prueba inicial:

Antes de lanzar la ejecución completa con todos los productos, comprueba que en la celda de prueba:

- Se abre el navegador correctamente y se aceptan las cookies.
- Se detectan los productos iniciales y el scroll funciona.
- `name` contiene el nombre real del artículo.
- `price` contiene el precio total en formato numérico correcto.
- `url` apunta correctamente a las fichas de AhorraMás.

Si esta prueba te devuelve los datos limpios de la leche, ya puedes cambiar a `TEST_MODE = False` y ejecutar el bloque completo de búsquedas.

In [15]:
import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, quote

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.ahorramas.com/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# MODO DE PRUEBA
# --------------------------------------------------
TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

MAX_SCROLLS = 30
SCROLL_PAUSE = 1.5
STABLE_ROUNDS_TO_STOP = 3

print("Búsquedas que se ejecutarán:", TERMS_TO_SCRAPE)

# --------------------------------------------------
# LOCALIZAR CARPETAS DEL PROYECTO
# --------------------------------------------------
def find_project_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path
    if current.name == "scrapers":
        return current.parent
    raise FileNotFoundError("No se ha podido localizar la raíz del proyecto.")

BASE_DIR = find_project_root()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", BASE_DIR)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

# --------------------------------------------------
# DRIVER Y CONFIGURACIÓN
# --------------------------------------------------
def start_ahorramas_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    driver.get(HOME_URL)
    return driver

def accept_ahorramas_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir')]",
        "//button[@id='onetrust-accept-btn-handler']"
    ]
    for xpath in xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)
            for btn in buttons:
                if btn.is_displayed():
                    btn.click()
                    print("Cookies aceptadas en AhorraMás.")
                    return
        except:
            pass
    print("No se encontró banner de cookies o ya estaba aceptado.")

# --------------------------------------------------
# BÚSQUEDA Y SCROLL
# --------------------------------------------------
def count_product_links(driver):
    # AhorraMás suele usar enlaces a productos con patrones específicos en su href
    links = driver.find_elements(By.CSS_SELECTOR, 'a[href*=".html"], a[class*="product"], div.product-tile a')
    hrefs = {link.get_attribute("href") for link in links if link.get_attribute("href")}
    return len(hrefs)

def search_ahorramas(driver, term):
    encoded_term = quote(term)
    search_url = f"https://www.ahorramas.com/search?q={encoded_term}"
    print(f"Navegando a: {search_url}")
    driver.get(search_url)
    time.sleep(4)

def load_all_ahorramas_products(driver, max_scrolls=MAX_SCROLLS, stable_rounds_to_stop=STABLE_ROUNDS_TO_STOP):
    previous_count = count_product_links(driver)
    stable_rounds = 0
    print("Productos iniciales detectados:", previous_count)

    for scroll_number in range(1, max_scrolls + 1):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE)
        
        current_count = count_product_links(driver)
        print(f"Scroll {scroll_number}: {previous_count} → {current_count}")

        if current_count > previous_count:
            previous_count = current_count
            stable_rounds = 0
        else:
            stable_rounds += 1

        if stable_rounds >= stable_rounds_to_stop:
            print("Carga finalizada (sin nuevos productos).")
            return current_count, "stable"

    return previous_count, "max_scrolls_reached"

# --------------------------------------------------
# EXTRACCIÓN DE DATOS
# --------------------------------------------------
def extract_ahorramas_products(driver, searched_item):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select('div.product-tile, li.grid-tile, div.grid-tile, div[class*="product-tile"]')
    if not cards:
        cards = soup.select('div[class*="product"], article')

    records = []
    seen_urls = set()
    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        text = card.get_text(" ", strip=True)
        
        link_elem = card.find("a", href=re.compile(r"\.html|/p/"))
        if not link_elem:
            link_elem = card.find("a", href=True)
            
        if not link_elem:
            continue
            
        href = link_elem.get("href")
        if not href or "search" in href or "cart" in href:
            continue
            
        url = urljoin(HOME_URL, href)
        if url in seen_urls:
            continue
        seen_urls.add(url)

        # Nombre del producto
        name = ""
        for tag in ["h3", "h2", "h4", "a", "span", "div"]:
            elem = card.find(tag, class_=re.compile("title|name|desc|link", re.I))
            if elem:
                txt = elem.get_text(" ", strip=True)
                if len(txt) > 4 and "€" not in txt:
                    name = txt
                    break
        
        if not name:
            link_text = link_elem.get_text(" ", strip=True)
            if len(link_text) > 4:
                name = link_text

        if not name:
            continue

        # Precio total
        price = None
        price_raw = None
        
        price_match = re.search(r"(\d+)[.,](\d{2})\s*€", text)
        if price_match:
            price_raw = f"{price_match.group(1)},{price_match.group(2)} €"
            price = float(f"{price_match.group(1)}.{price_match.group(2)}")
        else:
            price_match_alt = re.search(r"(\d+(?:[.,]\d+)?)\s*€", text)
            if price_match_alt:
                price_raw = price_match_alt.group(0)
                val_str = price_match_alt.group(1).replace(",", ".")
                val_float = float(val_str)
                if val_float > 50:
                    val_float = val_float / 100
                price = val_float

        if price is None or price > 50:
            continue

        # ID de producto
        id_match = re.search(r"/(\d+)\.html|/p/(\d+)", url)
        product_id = id_match.group(1) or id_match.group(2) if id_match else None

        # ==========================================
        # EXTRACCIÓN INTELIGENTE DE UNIDAD Y PRECIO/UNIDAD
        # ==========================================
        unit = None
        price_per_unit = None
        quantity_val = None

        name_lower = name.lower()
        
        # Buscar litros (ej: 1l, 1,5l, 500ml)
        l_match = re.search(r"(\d+(?:[.,]\d+)?)\s*(?:l|litros?)\b", name_lower)
        ml_match = re.search(r"(\d+)\s*ml\b", name_lower)
        
        # Buscar kilos / gramos (ej: 1kg, 1,5k, 500g, 800 g)
        kg_match = re.search(r"(\d+(?:[.,]\d+)?)\s*(?:kg|k)\b", name_lower)
        g_match = re.search(r"(\d+)\s*(?:g|gr)\b", name_lower)
        
        # Buscar unidades / docenas (ej: 12u, 6 unidades)
        u_match = re.search(r"(\d+)\s*(?:u|uds|unidades?)\b", name_lower)

        if l_match:
            quantity_val = float(l_match.group(1).replace(",", "."))
            unit = "LITRO"
        elif ml_match:
            quantity_val = float(ml_match.group(1)) / 1000.0
            unit = "LITRO"
        elif kg_match:
            quantity_val = float(kg_match.group(1).replace(",", "."))
            unit = "KILO"
        elif g_match:
            quantity_val = float(g_match.group(1)) / 1000.0
            unit = "KILO"
        elif u_match:
            quantity_val = float(u_match.group(1))
            unit = "UNIDAD"
            if quantity_val == 12 and "docena" in name_lower:
                unit = "DOCENA"

        # Calcular price_per_unit si tenemos cantidad válida
        if quantity_val and quantity_val > 0:
            price_per_unit = round(price / quantity_val, 4)

        records.append({
            "searched_item": searched_item,
            "supermarket": "AhorraMás",
            "product_id": product_id,
            "name": name,
            "quantity_raw": None,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": None,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)
    print(f"Productos extraídos para '{searched_item}': {len(df)}")
    return df
    
driver = start_ahorramas_driver()

try:
    time.sleep(3)
    accept_ahorramas_cookies(driver)

    test_term = "leche"
    search_ahorramas(driver, test_term)
    loaded_count, load_status = load_all_ahorramas_products(driver)
    
    df_test = extract_ahorramas_products(driver, test_term)

    print("\nEstado de carga:", load_status)
    print("Productos cargados en web:", loaded_count)
    print("Filas extraídas:", len(df_test))

    display(df_test.head(15))

finally:
    driver.quit()

Búsquedas que se ejecutarán: ['leche', 'huevos', 'manzana', 'plátano', 'tomate', 'arroz', 'pasta', 'pan', 'pollo', 'carne picada', 'salmón', 'yogur', 'queso', 'agua', 'zumo', 'atún', 'garbanzos', 'aceite', 'detergente', 'papel higiénico']
Proyecto: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project
Raw: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\raw
Processed: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\processed
Cookies aceptadas en AhorraMás.
Navegando a: https://www.ahorramas.com/search?q=leche
Productos iniciales detectados: 43
Scroll 1: 43 → 43
Scroll 2: 43 → 43
Scroll 3: 43 → 43
Carga finalizada (sin nuevos productos).
Productos extraídos para 'leche': 21

Estado de carga: stable
Productos cargados en web: 43
Filas extraídas: 21


,searched_item,supermarket,product_id,name,quantity_raw,price_raw,price,price_per_unit_raw,price_per_unit,unit,url,scraped_at
0,leche,AhorraMás,None,Leche Alipende 1l semidesnatada,None,"0,83 €",0.83,None,0.8300,LITRO,https://www.ahorramas.com/leche-alipende-1l-se...,2026-10-04T16:34:48
1,leche,AhorraMás,None,Leche Lauki 1l entera,None,"1,18 €",1.18,None,1.1800,LITRO,https://www.ahorramas.com/leche-lauki-1l-enter...,2026-10-04T16:34:48
2,leche,AhorraMás,None,Leche Alipende 1l entera,None,"0,96 €",0.96,None,0.9600,LITRO,https://www.ahorramas.com/leche-alipende-1l-en...,2026-10-04T16:34:48
3,leche,AhorraMás,None,Leche Alipende 1l desnatada,None,"0,81 €",0.81,None,0.8100,LITRO,https://www.ahorramas.com/leche-alipende-1l-de...,2026-10-04T16:34:48
4,leche,AhorraMás,None,Leche Puleva calcio 1l semidesnatada,None,"1,59 €",1.59,None,1.5900,LITRO,https://www.ahorramas.com/leche-puleva-calcio-...,2026-10-04T16:34:48
5,leche,AhorraMás,None,Leche Asturiana 1l semidesnatada,None,"1,17 €",1.17,None,1.1700,LITRO,https://www.ahorramas.com/leche-asturiana-1l-s...,2026-10-04T16:34:48
6,leche,AhorraMás,None,Leche Asturiana 1l entera,None,"1,29 €",1.29,None,1.2900,LITRO,https://www.ahorramas.com/leche-asturiana-1l-e...,2026-10-04T16:34:48
7,leche,AhorraMás,None,Leche Lauki 1l semidesnatada,None,"1,09 €",1.09,None,1.0900,LITRO,https://www.ahorramas.com/leche-lauki-1l-semid...,2026-10-04T16:34:48
8,leche,AhorraMás,None,Leche Pascual 1l entera,None,"1,39 €",1.39,None,1.3900,LITRO,https://www.ahorramas.com/leche-pascual-1l-ent...,2026-10-04T16:34:48
9,leche,AhorraMás,None,Leche Pascual 1l semidesnatada,None,"1,35 €",1.35,None,1.3500,LITRO,https://www.ahorramas.com/leche-pascual-1l-sem...,2026-10-04T16:34:48


In [16]:
all_results = []
summary = []

driver = start_ahorramas_driver()

try:
    time.sleep(3)
    accept_ahorramas_cookies(driver)

    for i, term in enumerate(SEARCH_TERMS, start=1):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(SEARCH_TERMS)}] Buscando en AhorraMás: {term}")

        start = time.monotonic()

        try:
            search_ahorramas(driver, term)
            loaded_count, load_status = load_all_ahorramas_products(driver)
            df_term = extract_ahorramas_products(driver, term)

            elapsed = round(time.monotonic() - start, 2)

            if not df_term.empty:
                all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "status": load_status,
                "products_loaded": loaded_count,
                "rows_extracted": len(df_term),
                "seconds": elapsed,
            })

            print(f"✓ {len(df_term)} filas extraídas en {elapsed} s")
            time.sleep(2)

        except Exception as exc:
            elapsed = round(time.monotonic() - start, 2)
            print(f"ERROR con '{term}': {type(exc).__name__}: {exc}")
            summary.append({
                "searched_item": term,
                "status": "error",
                "products_loaded": None,
                "rows_extracted": 0,
                "seconds": elapsed,
            })
            break

finally:
    driver.quit()

df_ahorramas = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
df_summary = pd.DataFrame(summary)
display(df_summary)
print("\nFilas raw totales:", len(df_ahorramas))

Cookies aceptadas en AhorraMás.

[1/20] Buscando en AhorraMás: leche
Navegando a: https://www.ahorramas.com/search?q=leche
Productos iniciales detectados: 43
Scroll 1: 43 → 43
Scroll 2: 43 → 43
Scroll 3: 43 → 43
Carga finalizada (sin nuevos productos).
Productos extraídos para 'leche': 21
✓ 21 filas extraídas en 12.69 s

[2/20] Buscando en AhorraMás: huevos
Navegando a: https://www.ahorramas.com/search?q=huevos
Productos iniciales detectados: 41
Scroll 1: 41 → 41
Scroll 2: 41 → 41
Scroll 3: 41 → 41
Carga finalizada (sin nuevos productos).
Productos extraídos para 'huevos': 19
✓ 19 filas extraídas en 11.78 s

[3/20] Buscando en AhorraMás: manzana
Navegando a: https://www.ahorramas.com/search?q=manzana
Productos iniciales detectados: 50
Scroll 1: 50 → 50
Scroll 2: 50 → 50
Scroll 3: 50 → 50
Carga finalizada (sin nuevos productos).
Productos extraídos para 'manzana': 20
✓ 20 filas extraídas en 15.7 s

[4/20] Buscando en AhorraMás: plátano
Navegando a: https://www.ahorramas.com/search?q=pl%

,searched_item,status,products_loaded,rows_extracted,seconds
0,leche,stable,43,21,12.69
1,huevos,stable,41,19,11.78
2,manzana,stable,50,20,15.70
3,plátano,stable,46,19,13.27
4,tomate,stable,46,20,13.51
5,arroz,stable,42,20,13.58
6,pasta,stable,42,20,13.50
7,pan,stable,45,23,13.61
8,pollo,stable,57,23,13.89
9,carne picada,stable,41,19,13.17



Filas raw totales: 398


In [17]:
# --------------------------------------------------
# GUARDAR CSV RAW
# --------------------------------------------------

if not df_ahorramas.empty:
    raw_path = RAW_DIR / "ahorramas_products_raw.csv"
    df_ahorramas.to_csv(raw_path, index=False, encoding="utf-8-sig")
    print("CSV raw guardado en:", raw_path)
else:
    print("No hay datos raw para guardar.")

CSV raw guardado en: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\raw\ahorramas_products_raw.csv


In [18]:
if not df_ahorramas.empty:
    df_clean = df_ahorramas.copy()

    # Deduplicar por product_id si existe, o por nombre y precio
    if "product_id" in df_clean.columns and df_clean["product_id"].notna().any():
        with_id = df_clean[df_clean["product_id"].notna()].drop_duplicates(subset=["product_id"])
        without_id = df_clean[df_clean["product_id"].isna()].drop_duplicates(subset=["name", "price"])
        df_clean = pd.concat([with_id, without_id], ignore_index=True).reset_index(drop=True)
    else:
        df_clean = df_clean.drop_duplicates(subset=["name", "price"]).reset_index(drop=True)

    processed_path = PROCESSED_DIR / "ahorramas_products.csv"
    df_clean.to_csv(processed_path, index=False, encoding="utf-8-sig")

    print("CSV procesado guardado:", processed_path)
    print("Productos únicos finales:", len(df_clean))
    display(df_clean.head(10))
else:
    print("No hay datos para procesar.")

CSV procesado guardado: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\processed\ahorramas_products.csv
Productos únicos finales: 393


,searched_item,supermarket,product_id,name,quantity_raw,price_raw,price,price_per_unit_raw,price_per_unit,unit,url,scraped_at
0,leche,AhorraMás,None,Leche Alipende 1l semidesnatada,None,"0,83 €",0.83,None,0.83,LITRO,https://www.ahorramas.com/leche-alipende-1l-se...,2026-10-04T16:35:31
1,leche,AhorraMás,None,Leche Lauki 1l entera,None,"1,18 €",1.18,None,1.18,LITRO,https://www.ahorramas.com/leche-lauki-1l-enter...,2026-10-04T16:35:31
2,leche,AhorraMás,None,Leche Alipende 1l entera,None,"0,96 €",0.96,None,0.96,LITRO,https://www.ahorramas.com/leche-alipende-1l-en...,2026-10-04T16:35:31
3,leche,AhorraMás,None,Leche Alipende 1l desnatada,None,"0,81 €",0.81,None,0.81,LITRO,https://www.ahorramas.com/leche-alipende-1l-de...,2026-10-04T16:35:31
4,leche,AhorraMás,None,Leche Puleva calcio 1l semidesnatada,None,"1,59 €",1.59,None,1.59,LITRO,https://www.ahorramas.com/leche-puleva-calcio-...,2026-10-04T16:35:31
5,leche,AhorraMás,None,Leche Asturiana 1l semidesnatada,None,"1,17 €",1.17,None,1.17,LITRO,https://www.ahorramas.com/leche-asturiana-1l-s...,2026-10-04T16:35:31
6,leche,AhorraMás,None,Leche Asturiana 1l entera,None,"1,29 €",1.29,None,1.29,LITRO,https://www.ahorramas.com/leche-asturiana-1l-e...,2026-10-04T16:35:31
7,leche,AhorraMás,None,Leche Lauki 1l semidesnatada,None,"1,09 €",1.09,None,1.09,LITRO,https://www.ahorramas.com/leche-lauki-1l-semid...,2026-10-04T16:35:31
8,leche,AhorraMás,None,Leche Pascual 1l entera,None,"1,39 €",1.39,None,1.39,LITRO,https://www.ahorramas.com/leche-pascual-1l-ent...,2026-10-04T16:35:31
9,leche,AhorraMás,None,Leche Pascual 1l semidesnatada,None,"1,35 €",1.35,None,1.35,LITRO,https://www.ahorramas.com/leche-pascual-1l-sem...,2026-10-04T16:35:31
